# 07b - Uji akhir model v1.1 pada TEST2 (dijalankan sekali)
Model: **bundel `bundle/fraud_xgb_mask_v1_1/`** (63 fitur, tanpa gender, usia, wilayah, dan tenure cacat). Data: `data/test2/` (seed 10, 20.000 nasabah baru, ID `U...`), yang belum pernah dipakai untuk apa pun. Integritasnya diverifikasi dengan SHA-256 saat dimuat.

**Aturan main**
* Tidak ada penyetelan setelah melihat hasil. Bila gagal, dilaporkan apa adanya.
* Model v1 ikut diuji pada data yang sama **hanya sebagai pembanding** (untuk memastikan v1.1 tidak lebih buruk), bukan untuk memilih ulang.
* Alur notebook ini sudah dilatih lebih dulu pada data DEV (mode latihan), sehingga bug tertangkap tanpa menyentuh TEST2.

**Kriteria lulus, ditetapkan sebelum TEST2 dimuat**
| # | Kriteria | Ambang |
|---|---|---|
| C1 | PR-AUC bersih v1.1 | >= 0,50 dan batas bawah CI95 >= 0,40 |
| C2 | PR-AUC saat 20% nilai hilang | >= 0,35 |
| C3 | Lebih baik dari aturan gate saja | batas bawah CI95 model > batas atas CI95 aturan |
| C4 | Presisi pita `high_risk` | >= 0,30 |
| C5 | Fairness (kesamaan peluang): rasio recall terbesar/terkecil, kelompok >= 30 hari fraud | <= 1,25 antar gender dan <= 1,5 antar usia |
| C6 | Tidak ada atribut terproteksi di model | `schema.json` tidak memuat gender, age_ord, region, tenure_months, hist_days |
| C7 | Tidak lebih buruk dari v1 | batas bawah CI95 selisih PR-AUC (v1.1 - v1) >= -0,05 |

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")      # notebooks use paths relative to the project root
print("working directory:", os.getcwd())

In [1]:
import os, sys, json, warnings
os.environ.setdefault("OMP_NUM_THREADS", "8"); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
sys.path.insert(0, os.getcwd()); import splits as S
from sklearn.metrics import average_precision_score, roc_auc_score

DRY = os.environ.get("DRY") == "1"                        # latihan: memakai DEV-OOT, TIDAK menyentuh TEST2
B11, B1, ART, L, W, ALERT, N_BOOT, SEED = "bundle/fraud_xgb_mask_v1_1", "bundle/fraud_xgb_mask_v1", "artifacts", S.LABEL, S.WEIGHT, 0.001, 200, 42
CRIT = dict(C1_pr=0.50, C1_ci_low=0.40, C2_miss20=0.35, C4_high_precision=0.30, C5_gender=1.25, C5_age=1.5, C5_min_fraud_days=30, C7_margin=-0.05)       # tetap
sys.path.insert(0, B11); from score import FraudScorer
s11, s1 = FraudScorer(B11), FraudScorer(B1)
F1 = s1.features; NUM1 = [f for f in F1 if f not in s1.cats]
print("v1.1:", len(s11.features), "fitur | v1:", len(F1), "fitur | ambang v1.1:", s11.schema["thresholds"] | {}, "| DRY =", DRY)

v1.1: 63 fitur | v1: 68 fitur | ambang v1.1: {'suspicious': 0.00247490918263793, 'high_risk': 0.04664194583892822, 'note': "berlaku pada 'score' (terkoreksi prior); suspicious = presisi >= 5% di DEV-OOT, high_risk = presisi >= 50%"} | DRY = False


In [2]:
if DRY:
    dev = S.load_dev(); test = S.dev_oot(dev).sample(60000, random_state=0).reset_index(drop=True)
else:
    test = S.load_test2(i_understand_this_is_the_final_evaluation=True)
    man = json.load(open(f"{S.TEST2}/MANIFEST.json")); print("hash diverifikasi:", list(man["sha256"]))
print(f"{len(test):,} baris | nasabah {test.customer_id.nunique():,} | nasabah fraud {test.loc[test[L] == 1, 'customer_id'].nunique()} | baris fraud {int(test[L].sum()):,} | prevalensi {np.average(test[L], weights=test[W]):.4%}")
y, w = test[L].values, test[W].values

hash diverifikasi: ['daydetect.parquet', 'customers.csv', 'fraud_events.csv']
195,769 baris | nasabah 19,477 | nasabah fraud 594 | baris fraud 2,053 | prevalensi 0.0318%


In [3]:
def rule_gate(d):
    h = ((d.new_device_7 > 0).astype(int) + (d.cred_change_3 > 0) + (d.ftp_large_7 > 0) + (d.night_share_7 > 0.3) + (d.geo_mismatch_share_7 > 0.5) + (d.out_to_balance_max_3d > 0.5) + (d.new_payee_cnt_7 >= 2))
    return h.values + 0.01 * d.out_to_balance_1.values
def metrics(y, s, w, alert=ALERT):
    o = np.argsort(-s); ys, ws = y[o], w[o]; k = np.searchsorted(np.cumsum(ws), alert * ws.sum()); tp = (ys[:k] * ws[:k]).sum()
    return dict(PR_AUC=average_precision_score(y, s, sample_weight=w), ROC_AUC=roc_auc_score(y, s, sample_weight=w), recall_at_alert=tp / (ys * ws).sum(), precision_at_alert=tp / ws[:k].sum())
X = test[F1]                                                      # v1.1 memakai himpunan bagian dari kolom yang sama
r11, r1 = s11.score_df(X, reasons=False), s1.score_df(X, reasons=False)
scores = {"v1.1 (63 fitur, bundel)": r11.score_raw.values, "v1 (68 fitur, bundel)": r1.score_raw.values, "Aturan gate saja": rule_gate(test)}
tab = pd.DataFrame({k: metrics(y, s, w) for k, s in scores.items()}).T; champ = "v1.1 (63 fitur, bundel)"; tab

,PR_AUC,ROC_AUC,recall_at_alert,precision_at_alert
"v1.1 (63 fitur, bundel)",0.733506,0.998548,0.855821,0.272108
"v1 (68 fitur, bundel)",0.752088,0.998546,0.861666,0.274874
Aturan gate saja,0.007605,0.843079,0.049683,0.015850


In [4]:
codes, n_c = pd.factorize(test.customer_id)[0], test.customer_id.nunique(); rng = np.random.default_rng(SEED); B = {k: [] for k in scores}
for _ in range(N_BOOT):
    wb = w * np.bincount(rng.integers(0, n_c, n_c), minlength=n_c)[codes]
    for k, s in scores.items(): B[k].append(average_precision_score(y, s, sample_weight=wb))
B = pd.DataFrame(B)
ci = pd.DataFrame({"PR_AUC": tab.PR_AUC, "CI95_low": B.quantile(.025), "CI95_high": B.quantile(.975)}).round(4); display(ci)
d_v = (B[champ] - B["v1 (68 fitur, bundel)"]).quantile([.025, .5, .975]).round(4); print("Selisih PR-AUC v1.1 - v1 [2.5%, median, 97.5%]:", d_v.values)

,PR_AUC,CI95_low,CI95_high
"v1.1 (63 fitur, bundel)",0.7335,0.6755,0.7869
"v1 (68 fitur, bundel)",0.7521,0.6934,0.8080
Aturan gate saja,0.0076,0.0056,0.0137


Selisih PR-AUC v1.1 - v1 [2.5%, median, 97.5%]: [-0.0393 -0.0168  0.0008]


### Pita risiko (ambang beku dari DEV-OOT), tangkapan per tipe fraud

In [5]:
b = r11.band.values; d = pd.DataFrame({"band": b, "w": w, "fraud": w * y}).groupby("band").sum().reindex(["genuine", "suspicious", "high_risk"]).fillna(0)
d["share_of_traffic"] = d.w / d.w.sum(); d["precision"] = d.fraud / d.w.replace(0, np.nan); d["share_of_fraud_caught"] = d.fraud / d.fraud.sum()
display(d[["share_of_traffic", "precision", "share_of_fraud_caught"]].style.format("{:.3%}"))
pos = test[y == 1].assign(band=b[y == 1])
display(pd.crosstab([pos._fraud_type, pos._stealth], pos.band, normalize="index").reindex(columns=["genuine", "suspicious", "high_risk"]).fillna(0).round(3))

,share_of_traffic,precision,share_of_fraud_caught
band,,,
genuine,99.533%,0.002%,6.283%
suspicious,0.421%,1.210%,16.025%
high_risk,0.046%,53.855%,77.691%


band                  genuine  suspicious  high_risk
_fraud_type _stealth                                
ato         0           0.018       0.164      0.818
            1           0.294       0.422      0.284
mule        0           0.034       0.069      0.898
            1           0.110       0.172      0.718
scam        0           0.003       0.133      0.865
            1           0.042       0.240      0.718

### Ketahanan terhadap data tidak lengkap

In [6]:
def mask_cells(d, rate, rng):
    d = d.copy()
    for c in F1: d[c] = d[c].mask(rng.random(len(d)) < rate)
    return d
def blank(d, cols):
    d = d.copy()
    for c in cols: d[c] = d[c].mask(np.ones(len(d), bool))
    return d
def noise(d, sg, rng):
    d = d.copy(); d[NUM1] = d[NUM1].astype("float64") * np.exp(rng.normal(0, sg, (len(d), len(NUM1)))); return d
r = np.random.default_rng(400); Xc = X.copy()
for c in s1.cats: Xc[c] = pd.Categorical(Xc[c].astype(object).where(Xc[c].astype(object).isin(s1.cats[c])), categories=s1.cats[c])
SC = {"clean": Xc, "miss20": mask_cells(Xc, .2, r), "tanpa_payee": blank(Xc, s1.groups["payee"]), "noise15": noise(Xc, .15, r), "miss20+noise15": noise(mask_cells(Xc, .2, r), .15, r)}
rows = {}
for sname, dd in SC.items():
    a, bb = s11.score_df(dd, reasons=False).score_raw.values, s1.score_df(dd, reasons=False).score_raw.values
    rows[sname] = dict(v1_1_PR=metrics(y, a, w)["PR_AUC"], v1_PR=metrics(y, bb, w)["PR_AUC"], v1_1_recall_alert=metrics(y, a, w)["recall_at_alert"])
    if sname == "miss20": s_miss = a
rob = pd.DataFrame(rows).T.round(4); display(rob)
wb_ci = []
for _ in range(N_BOOT):
    wb = w * np.bincount(rng.integers(0, n_c, n_c), minlength=n_c)[codes]; wb_ci.append(average_precision_score(y, s_miss, sample_weight=wb))
miss_ci = np.quantile(wb_ci, [.025, .975]).round(4); print("PR-AUC v1.1 saat 20% hilang, CI95:", miss_ci)

,v1_1_PR,v1_PR,v1_1_recall_alert
clean,0.7335,0.7521,0.8558
miss20,0.5576,0.5821,0.7107
tanpa_payee,0.4659,0.5018,0.6006
noise15,0.4623,0.4675,0.5884
miss20+noise15,0.3354,0.3198,0.4481


PR-AUC v1.1 saat 20% hilang, CI95: [0.4944 0.6245]


### Fairness (kesamaan peluang) pada ambang alert 0,1% traffic

In [7]:
s = scores[champ]; thr = np.sort(s)[::-1][np.searchsorted(np.cumsum(w[np.argsort(-s)]), ALERT * w.sum())]; flag = s >= thr; fair = []
for col in ("gender", "age_ord", "region"):
    for v, gdf in test.assign(f=flag).groupby(col):
        gp = gdf[gdf[L] == 1]; fair.append(dict(group=f"{col}={v}", flag_rate=np.average(gdf.f, weights=gdf[W]), recall=gp.f.mean() if len(gp) else np.nan, n_fraud_days=len(gp)))
fair = pd.DataFrame(fair).set_index("group"); display(fair.round(4))
def ratio(p, col="recall"):
    g_ = fair[fair.index.str.startswith(p) & (fair.n_fraud_days >= CRIT["C5_min_fraud_days"])][col]; return g_.max() / g_.min()
print(f"rasio recall terbesar/terkecil: gender {ratio('gender'):.2f} | usia {ratio('age_ord'):.2f} | wilayah {ratio('region'):.2f} (info)")

,flag_rate,recall,n_fraud_days
group,,,
gender=F,0.0010,0.8612,1131
gender=M,0.0010,0.8492,922
age_ord=0,0.0012,0.8647,584
age_ord=1,0.0010,0.8512,652
age_ord=2,0.0010,0.8440,327
age_ord=3,0.0008,0.8528,265
age_ord=4,0.0009,0.8667,225
region=Bali,0.0012,0.9216,51
region=Banten,0.0010,0.8741,135


rasio recall terbesar/terkecil: gender 1.01 | usia 1.03 | wilayah 1.18 (info)


## Keputusan lulus-gagal

In [8]:
c_pr, c_lo = ci.loc[champ, "PR_AUC"], ci.loc[champ, "CI95_low"]; prot = {"gender", "age_ord", "region", "tenure_months", "hist_days"}
checks = {
  "C1 PR-AUC bersih >= 0,50 dan CI bawah >= 0,40": (c_pr >= CRIT["C1_pr"] and c_lo >= CRIT["C1_ci_low"], f"{c_pr:.3f} (CI {c_lo:.3f}-{ci.loc[champ, 'CI95_high']:.3f})"),
  "C2 PR-AUC 20% hilang >= 0,35": (rob.loc["miss20", "v1_1_PR"] >= CRIT["C2_miss20"], f"{rob.loc['miss20', 'v1_1_PR']:.3f} (CI {miss_ci[0]:.3f}-{miss_ci[1]:.3f})"),
  "C3 CI bawah v1.1 > CI atas aturan gate": (ci.loc[champ, "CI95_low"] > ci.loc["Aturan gate saja", "CI95_high"], f"{ci.loc[champ, 'CI95_low']:.3f} vs {ci.loc['Aturan gate saja', 'CI95_high']:.3f}"),
  "C4 presisi high_risk >= 0,30": (d.loc["high_risk", "precision"] >= CRIT["C4_high_precision"] if d.loc["high_risk", "w"] > 0 else False, f"{d.loc['high_risk', 'precision']:.3f}"),
  "C5 fairness recall: gender <= 1,25 dan usia <= 1,5": (ratio("gender") <= CRIT["C5_gender"] and ratio("age_ord") <= CRIT["C5_age"], f"gender {ratio('gender'):.2f}, usia {ratio('age_ord'):.2f}"),
  "C6 tanpa atribut terproteksi / tenure cacat": (not (prot & set(s11.features)) and s11.schema.get("protected_attributes_in_model") == [], f"{len(s11.features)} fitur"),
  "C7 tidak lebih buruk dari v1 (CI bawah selisih >= -0,05)": (d_v.iloc[0] >= CRIT["C7_margin"], f"median {d_v.iloc[1]:+.3f} (CI {d_v.iloc[0]:+.3f} .. {d_v.iloc[2]:+.3f})"),
}
verdict = pd.DataFrame({k: dict(lulus="LULUS" if ok else "GAGAL", nilai=v) for k, (ok, v) in checks.items()}).T; display(verdict)
print("HASIL AKHIR:", "SEMUA KRITERIA LULUS" if all(ok for ok, _ in checks.values()) else "ADA KRITERIA GAGAL - lihat tabel", "(LATIHAN, bukan TEST2)" if DRY else "")

,lulus,nilai
"C1 PR-AUC bersih >= 0,50 dan CI bawah >= 0,40",LULUS,0.734 (CI 0.675-0.787)
"C2 PR-AUC 20% hilang >= 0,35",LULUS,0.558 (CI 0.494-0.625)
C3 CI bawah v1.1 > CI atas aturan gate,LULUS,0.675 vs 0.014
"C4 presisi high_risk >= 0,30",LULUS,0.539
"C5 fairness recall: gender <= 1,25 dan usia <= 1,5",LULUS,"gender 1.01, usia 1.03"
C6 tanpa atribut terproteksi / tenure cacat,LULUS,63 fitur
"C7 tidak lebih buruk dari v1 (CI bawah selisih >= -0,05)",LULUS,median -0.017 (CI -0.039 .. +0.001)


HASIL AKHIR: SEMUA KRITERIA LULUS 


In [9]:
if DRY: print("mode latihan: tidak menyimpan apa pun")
else:
    out = dict(test_manifest_sha=man["sha256"], criteria=CRIT, pr_auc=ci.to_dict(), table=tab.round(4).to_dict(), bands=d[["share_of_traffic", "precision", "share_of_fraud_caught"]].round(5).to_dict(),
               robustness=rob.to_dict(), fairness=fair.round(5).to_dict(), v11_minus_v1=d_v.to_dict(), verdict={k: dict(ok=bool(ok), value=v) for k, (ok, v) in checks.items()})
    json.dump(out, open(f"{ART}/test2_results.json", "w"), indent=1, default=str)
    card = open(f"{B11}/MODEL_CARD.md").read().split("## Hasil TEST2")[0]
    card += "## Hasil TEST2 (data baru, seed 10; dijalankan sekali)\n" + f"* PR-AUC bersih {c_pr:.3f} (CI95 {c_lo:.3f}-{ci.loc[champ, 'CI95_high']:.3f}); 20% nilai hilang {rob.loc['miss20', 'v1_1_PR']:.3f}; tanpa payee {rob.loc['tanpa_payee', 'v1_1_PR']:.3f}; aturan gate saja {ci.loc['Aturan gate saja', 'PR_AUC']:.3f}; v1 pada data yang sama {ci.loc['v1 (68 fitur, bundel)', 'PR_AUC']:.3f}.\n"
    card += "* Kriteria: " + "; ".join(f"{k.split(' ')[0]} {'lulus' if ok else 'GAGAL'}" for k, (ok, _) in checks.items()) + ".\n* Detail: `artifacts/test2_results.json`, notebook `07b_final_test_v1_1.ipynb`.\n"
    open(f"{B11}/MODEL_CARD.md", "w").write(card); print("tersimpan: artifacts/test2_results.json; MODEL_CARD.md diperbarui")

tersimpan: artifacts/test2_results.json; MODEL_CARD.md diperbarui
